# Clase 9: Ingeniería de Datos y Clasificación de Billetes (PYG)

En esta sesión construimos un sistema de visión artificial **desde cero** para una terminal de depósito automático de Guaraníes. No hay un dataset descargable: ustedes son los ingenieros encargados de recolectar los datos, entrenar el modelo y desplegarlo.

Este notebook introduce un concepto fundamental del ML profesional: **la calidad del modelo depende más de los datos que de la arquitectura.**

---

## ¿Por qué este problema es difícil?

A diferencia de la Clase 6 (dataset NEU con imágenes controladas de laboratorio), aquí enfrentamos **el mundo real**:

| Desafío | ¿Qué pasa en la terminal? |
|---|---|
| **Desgaste** | Un billete de 2.000 Gs. circulado 5 años se ve diferente al nuevo |
| **Iluminación** | Luz fluorescente de oficina vs. luz solar directa |
| **Orientación** | El usuario puede insertar el billete al revés o torcido |
| **Oclusión** | Los dedos del usuario aparecen parcialmente en la imagen |
| **Variedad de cámaras** | Cada celular del grupo tiene diferente sensor y balance de blancos |

Cada uno de estos factores puede confundir a un modelo mal entrenado y hacer que una terminal acepte el billete equivocado.

---

## El enfoque Data-Centric

Existe un debate histórico en IA entre dos filosofías:

- **Model-Centric:** ¿El modelo comete errores? Hacelo más grande o cambiá la arquitectura.
- **Data-Centric:** ¿El modelo comete errores? Mejorar los datos primero.

Andrew Ng (cofundador de Google Brain, Coursera) demostró que en la mayoría de los proyectos industriales reales, **mejorar los datos supera en impacto a mejorar el modelo**. En esta clase adoptamos esa filosofía: primero construimos un dataset de calidad, luego entrenamos.

---

## Estructura del notebook

```
Bloque 0  →  Reproducibilidad (semilla aleatoria)
Bloque 1  →  Configuración de hardware (GPU/CPU)
Bloque 2  →  Instrucciones de recolección de datos
Bloque 3  →  Verificación y exploración del dataset
Bloque 4  →  Transformaciones y Data Augmentation
Bloque 5  →  Transfer Learning con MobileNetV2
Bloque 6  →  Entrenamiento con early stopping
Bloque 7  →  Curvas de aprendizaje
Bloque 8  →  Evaluación: matriz de confusión y métricas
Bloque 9  →  Simulador de terminal de depósito
Bloque 10 →  Exportación del modelo
```

---

## Bloque 0: Reproducibilidad

Las redes neuronales usan números aleatorios durante la inicialización de pesos y en el Dropout. Sin fijar una semilla, dos ejecuciones del mismo código pueden dar resultados diferentes. Esto hace imposible comparar experimentos o depurar errores.

**Regla profesional:** todo experimento de ML comienza fijando la semilla.

In [ ]:
import random
import numpy as np
import torch

SEED = 42

def fijar_semilla(seed: int):
    """Fija la semilla aleatoria en todos los módulos relevantes."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

fijar_semilla(SEED)
print(f"Semilla fijada: {SEED}")
print(f"PyTorch versión: {torch.__version__}")

---

## Bloque 1: Configuración de hardware

PyTorch puede ejecutar cálculos en dos lugares:
- **CPU (procesador):** versátil, lento para tensores grandes
- **GPU (tarjeta de video):** miles de núcleos en paralelo, ideal para operaciones matriciales

El entrenamiento de una red neuronal es esencialmente **multiplicación de matrices a escala masiva**. Una GPU puede hacer en segundos lo que la CPU tarda minutos.

**Error frecuente:** si un tensor está en CPU y el modelo en GPU (o viceversa), PyTorch lanza `RuntimeError: Expected all tensors to be on the same device`. Por eso usamos una variable `device` global y enviamos todo al mismo lugar.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import json
import numpy as np
from PIL import Image
from datetime import datetime

# ── Detección de hardware ─────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de cómputo: {device}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU detectada: {gpu.name}")
    print(f"VRAM disponible: {gpu.total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: No se detectó GPU. El entrenamiento será más lento.")
    print("Si estás en la Predator, verificá que los drivers CUDA estén instalados.")

---

## Bloque 2: Recolección del dataset — Instrucciones de campo

Este bloque no tiene código: tiene las instrucciones de ingeniería para construir el dataset. Léanlas con atención antes de salir a fotografiar.

### Estructura de carpetas requerida

```
dataset_billetes/
├── train/          ← 80% de las fotos
│   ├── 2k/
│   ├── 5k/
│   ├── 10k/
│   ├── 20k/
│   ├── 50k/
│   └── 100k/
└── val/            ← 20% de las fotos
    ├── 2k/
    ├── 5k/
    ├── 10k/
    ├── 20k/
    ├── 50k/
    └── 100k/
```

**Importante:** Los nombres de carpeta (`2k`, `5k`, etc.) serán leídos automáticamente por PyTorch como nombres de clase. No usen espacios ni caracteres especiales.

### Protocolo de captura (mínimo 15 fotos por denominación)

El objetivo es que el dataset contenga **toda la variabilidad que verá la terminal en producción**. Para lograrlo, cada denominación debe tener fotos en estas condiciones:

| Variación | ¿Qué simula? | ¿Cómo lograrlo? |
|---|---|---|
| **Fondos distintos** | Diferentes superficies de la ranura | Madera, tela, palma de la mano, papel blanco |
| **Iluminación** | Diferentes horarios y locales | Luz fluorescente, ventana, sombra parcial |
| **Orientación** | El usuario inserta el billete torcido | Rotá el billete 15°, 45°, 90° |
| **Cara del billete** | El billete puede entrar al revés | Fotografiá ambas caras |
| **Oclusión leve** | Los dedos aparecen en la imagen | Sostené el billete con dos dedos visibles |
| **Perspectiva** | La cámara no siempre es cenital | Incliná el celular 30° respecto al billete |
| **Desgaste** | Billetes de baja denominación circulados | Buscá billetes viejos y arrugados |

### La regla de oro

> **"Garbage In, Garbage Out"**: Si todas las fotos de entrenamiento muestran el billete perfectamente centrado sobre fondo blanco bajo luz de ventana, la terminal fallará en cualquier otra condición. La diversidad del dataset determina la robustez del sistema.

---

## Bloque 3: Verificación y exploración del dataset

Antes de entrenar, verificamos que la estructura de carpetas sea correcta y que el dataset esté balanceado. Un dataset muy desbalanceado (muchas fotos de 100k y pocas de 2k) hará que el modelo sea más preciso en denominaciones con más datos.

**Regla de ingeniería:** nunca entrenés sobre datos que no viste. Este bloque muestra una muestra de cada clase para que el grupo valide que las imágenes están bien etiquetadas.

In [ ]:
# ── Configuración de rutas ────────────────────────────────────────────────────
DATASET_DIR = './dataset_billetes'
TRAIN_DIR   = os.path.join(DATASET_DIR, 'train')
VAL_DIR     = os.path.join(DATASET_DIR, 'val')

# ── Verificación de estructura ────────────────────────────────────────────────
errores = []
for split in ['train', 'val']:
    split_path = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_path):
        errores.append(f"No se encontró la carpeta: '{split_path}'")

if errores:
    for e in errores:
        print(f"ERROR: {e}")
    print("\nCreá la estructura de carpetas según las instrucciones del Bloque 2 y volvé a ejecutar.")
else:
    clases = sorted(os.listdir(TRAIN_DIR))
    print(f"Dataset encontrado. Clases detectadas ({len(clases)}): {clases}")

    # ── Estadísticas de distribución ──────────────────────────────────────────
    print(f"\n{'Denominación':<15} {'Train':>8} {'Val':>8} {'Total':>8}")
    print('-' * 43)
    totales = {'train': 0, 'val': 0}
    for clase in clases:
        n_train = len(os.listdir(os.path.join(TRAIN_DIR, clase)))
        n_val   = len(os.listdir(os.path.join(VAL_DIR,   clase)))
        totales['train'] += n_train
        totales['val']   += n_val
        print(f"{clase:<15} {n_train:>8} {n_val:>8} {n_train+n_val:>8}")
    print('-' * 43)
    print(f"{'TOTAL':<15} {totales['train']:>8} {totales['val']:>8} {sum(totales.values()):>8}")

    # ── Advertencia de desbalance ─────────────────────────────────────────────
    conteos = [len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in clases]
    if max(conteos) / max(min(conteos), 1) > 2:
        print("\nADVERTENCIA: El dataset está desbalanceado (razón max/min > 2x).")
        print("Considerá agregar más fotos a las clases con menor cantidad.")
    else:
        print("\nDataset balanceado.")

    # ── Visualización de muestras ──────────────────────────────────────────────
    n_clases = len(clases)
    fig, axes = plt.subplots(2, max(n_clases // 2, 1), figsize=(15, 6))
    fig.suptitle('Muestra de cada denominación — verificación de etiquetas', fontsize=13)

    for ax, clase in zip(axes.flat, clases):
        clase_path = os.path.join(TRAIN_DIR, clase)
        archivos   = [f for f in os.listdir(clase_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if archivos:
            img = Image.open(os.path.join(clase_path, archivos[0])).convert('RGB')
            ax.imshow(img)
            n = len(archivos)
            ax.set_title(f'{clase}\n({n} imágenes train)', fontsize=10)
        else:
            ax.set_title(f'{clase}\n(SIN IMÁGENES)', color='red', fontsize=10)
        ax.axis('off')

    # Ocultar ejes vacíos si el número de clases es impar
    for ax in axes.flat[n_clases:]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

---

## Bloque 4: Transformaciones y Data Augmentation

### El problema de los datasets pequeños

Con 15 fotos por clase, una red neuronal puede **memorizar** las imágenes exactas en lugar de aprender los rasgos generales del billete. Si la única foto de 100k en entrenamiento tiene un fondo de madera marrón, el modelo podría aprender a clasificar como 100k todo lo que tenga ese fondo, no el billete en sí.

### La solución: Data Augmentation

En cada época, en lugar de mostrar siempre la misma foto, aplicamos **transformaciones aleatorias** que generan versiones distintas de cada imagen. La red nunca ve exactamente la misma imagen dos veces, lo que la obliga a aprender características más generales.

### ¿Por qué los valores de normalización son esos?

Usamos `mean=[0.485, 0.456, 0.406]` y `std=[0.229, 0.224, 0.225]` porque son las estadísticas calculadas sobre el dataset **ImageNet** (1.4 millones de imágenes). Como vamos a usar MobileNetV2 que fue entrenado en ImageNet, nuestras imágenes deben estar en el mismo rango de valores que las imágenes con las que ese modelo fue entrenado. Es como hablar el mismo "idioma numérico" que ya conoce la red.

### ¿Por qué no aumentar los datos de validación?

La validación debe simular exactamente las condiciones de producción. Si rotamos aleatoriamente las imágenes de validación, estaríamos midiendo el rendimiento en condiciones artificiales, no reales. **El set de validación siempre es la foto original, sin modificar.**

In [ ]:
# ── Hiperparámetros de preprocesamiento ───────────────────────────────────────
IMG_SIZE   = 224   # Tamaño de entrada estándar de MobileNetV2
BATCH_SIZE = 16    # Lotes pequeños por el dataset reducido

# Estadísticas de ImageNet (necesarias para Transfer Learning)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Transformaciones de entrenamiento (CON aumentación) ───────────────────────
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Simula el billete entrando torcido a la ranura
    transforms.RandomRotation(degrees=30),

    # Simula diferentes distancias a la cámara (zoom in/out)
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),

    # El billete puede entrar con la cara del reverso
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),

    # Variaciones de iluminación (diferentes locales y horarios)
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),

    # Simula sombras o manchas localizadas en el billete
    transforms.RandomGrayscale(p=0.05),

    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ── Transformaciones de validación (SIN aumentación) ─────────────────────────
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ── Carga de datasets ─────────────────────────────────────────────────────────
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform_train)
val_dataset   = datasets.ImageFolder(root=VAL_DIR,   transform=transform_val)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Imágenes de entrenamiento: {len(train_dataset)}")
print(f"Imágenes de validación:    {len(val_dataset)}")

# Verificar shape de un batch
imgs, labels = next(iter(train_loader))
print(f"\nShape de un batch: {imgs.shape}  → [Batch, Canales, Alto, Ancho]")
print(f"  → {BATCH_SIZE} billetes, 3 canales RGB, {IMG_SIZE}×{IMG_SIZE} píxeles")

---

## Bloque 5: Transfer Learning con MobileNetV2

### ¿Qué es Transfer Learning?

Imaginá que querés enseñarle a un médico experto en radiología a leer tomografías de tórax. No lo entrenás desde cero: ya sabe anatomía, ya sabe interpretar imágenes médicas, ya sabe lo que es una anomalía. Solo necesita aprender las especificidades del nuevo tipo de imagen.

Con redes neuronales pasa exactamente lo mismo. **MobileNetV2** fue entrenada por Google durante semanas en 1.4 millones de imágenes. Ya aprendió a detectar bordes, texturas, formas, colores, patrones. Nosotros no necesitamos enseñarle todo eso de nuevo: solo le enseñamos a reconocer qué combinación de esas características corresponde a cada denominación de Guaraní.

### Estrategia: Feature Extraction

Congelamos todos los parámetros de MobileNetV2 (no se actualizan durante el entrenamiento) y reemplazamos solo la **capa final de clasificación** por una nueva que tiene tantas salidas como denominaciones tenemos.

```
MobileNetV2 (congelada)
├── Capas convolucionales profundas  ← "Los ojos" (no tocar)
│   Detectan: bordes, texturas, colores, formas
└── Clasificador original (1000 clases)  ← Reemplazamos esto
    ↓
    Nuevo clasificador (N denominaciones)
    ├── Linear(1280 → 512)
    ├── ReLU
    ├── Dropout(0.4)
    └── Linear(512 → NUM_CLASSES)
```

### ¿Por qué MobileNetV2 y no ResNet?

En una terminal de servicios real, la velocidad de inferencia importa tanto como la precisión. MobileNetV2 fue diseñada específicamente para dispositivos con recursos limitados (celulares, terminales embebidas): es entre 5x y 10x más rápida que ResNet-50 con una pérdida de precisión mínima.

In [ ]:
def crear_modelo_pyg(num_clases: int):
    """
    Construye un clasificador de billetes basado en MobileNetV2 pre-entrenado.
    
    Estrategia: Feature Extraction
    - Capas convolucionales de MobileNetV2: CONGELADAS (no se entrenan)
    - Clasificador final: REEMPLAZADO y entrenado desde cero
    
    Args:
        num_clases: Número de denominaciones (ej: 6 para 2k/5k/10k/20k/50k/100k)
    
    Returns:
        Modelo PyTorch listo para entrenar
    """
    # Cargar MobileNetV2 con pesos pre-entrenados en ImageNet
    modelo = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # Congelar TODAS las capas convolucionales
    # requires_grad=False le dice a PyTorch: "no calcules gradientes aquí"
    # (= no actualices estos pesos durante el entrenamiento)
    for param in modelo.parameters():
        param.requires_grad = False

    # Reemplazar el clasificador final
    # model.classifier[1] es la última capa Linear de MobileNetV2
    # in_features=1280 es el número de características que entrega MobileNet
    modelo.classifier[1] = nn.Sequential(
        nn.Linear(1280, 512),
        nn.ReLU(),
        nn.Dropout(p=0.4),    # Evita overfitting con datasets pequeños
        nn.Linear(512, num_clases)
        # Sin Softmax al final: CrossEntropyLoss ya lo incluye internamente
    )

    return modelo


# ── Instanciar y enviar a GPU ─────────────────────────────────────────────────
model = crear_modelo_pyg(NUM_CLASSES).to(device)

# ── Resumen de parámetros ─────────────────────────────────────────────────────
params_total       = sum(p.numel() for p in model.parameters())
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_congelados  = params_total - params_entrenables

print("Arquitectura: MobileNetV2 + Clasificador personalizado")
print(f"Parámetros totales:       {params_total:>12,}")
print(f"Parámetros congelados:    {params_congelados:>12,}  ← MobileNetV2 (no se tocan)")
print(f"Parámetros entrenables:   {params_entrenables:>12,}  ← Solo el nuevo clasificador")
pct = params_entrenables / params_total * 100
print(f"\nEntrenamos solo el {pct:.1f}% del modelo total.")
print("Esto hace el entrenamiento muy rápido incluso con pocos datos.")

---

## Bloque 6: Entrenamiento

### Los 5 pasos de cada iteración de entrenamiento

Cada vez que el modelo ve un lote (batch) de imágenes, ocurren 5 pasos en orden:

1. **`zero_grad()`** — Limpiar los gradientes del paso anterior (si no se limpian, se acumulan y los cálculos se corrompen)
2. **Forward pass** — La imagen pasa por todas las capas y el modelo genera una predicción
3. **Calcular pérdida** — `CrossEntropyLoss` mide qué tan equivocada fue la predicción
4. **`backward()`** — Backpropagation: calcula cuánto contribuyó cada peso al error
5. **`optimizer.step()`** — Ajusta los pesos en la dirección que reduce el error

Este ciclo se repite miles de veces hasta que el modelo converge.

### ¿Qué es el Learning Rate?

El Learning Rate (LR) controla **qué tan grandes son los ajustes** en cada paso. Si es muy alto, los pesos "saltan" demasiado y nunca convergen. Si es muy bajo, el entrenamiento es muy lento. El scheduler `ReduceLROnPlateau` lo reduce automáticamente cuando el modelo deja de mejorar.

### Early Stopping

Si la pérdida de validación no mejora en `PATIENCE` épocas consecutivas, detenemos el entrenamiento. Esto evita que el modelo haga overfitting sobreentrenándose en los datos de train. El mejor modelo (el del momento de menor val_loss) se guarda automáticamente.

In [ ]:
# ── Hiperparámetros ───────────────────────────────────────────────────────────
EPOCHS        = 30
LEARNING_RATE = 1e-3
PATIENCE      = 6     # Épocas sin mejora antes de detener
RUTA_CKPT     = 'mejor_modelo_billetes.pth'

# ── Función de pérdida y optimizador ─────────────────────────────────────────
# Solo optimizamos los parámetros entrenables (el nuevo clasificador)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4  # Regularización L2 adicional
)
# Reduce LR a la mitad si val_loss no mejora en 3 épocas
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

# ── Historial de métricas ─────────────────────────────────────────────────────
historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# ── Variables de early stopping ───────────────────────────────────────────────
mejor_val_loss    = float('inf')
epocas_sin_mejora = 0


def evaluar(loader):
    """Evalúa el modelo en un DataLoader. Retorna (loss_promedio, accuracy)."""
    model.eval()
    total_loss, correctos, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correctos += (preds == labels).sum().item()
            total     += labels.size(0)
    return total_loss / len(loader), correctos / total


# ── Ciclo de entrenamiento ────────────────────────────────────────────────────
print('Iniciando entrenamiento...')
print(f'{"Época":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8} | {"LR":>8}')
print('-' * 65)

for epoch in range(1, EPOCHS + 1):

    # ── Fase de entrenamiento ─────────────────────────────────────────────
    model.train()
    train_loss_acc, correctos_train, total_train = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()             # Paso 1: limpiar gradientes
        outputs = model(imgs)             # Paso 2: forward pass
        loss    = criterion(outputs, labels)  # Paso 3: calcular pérdida
        loss.backward()                   # Paso 4: backpropagation

        # Gradient clipping: evita que los gradientes "exploten"
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()                  # Paso 5: actualizar pesos

        train_loss_acc += loss.item()
        _, preds = torch.max(outputs, 1)
        correctos_train += (preds == labels).sum().item()
        total_train     += labels.size(0)

    train_loss = train_loss_acc / len(train_loader)
    train_acc  = correctos_train / total_train

    # ── Fase de validación ────────────────────────────────────────────────
    val_loss, val_acc = evaluar(val_loader)
    scheduler.step(val_loss)

    # Registrar historial
    historial['train_loss'].append(train_loss)
    historial['val_loss'].append(val_loss)
    historial['train_acc'].append(train_acc)
    historial['val_acc'].append(val_acc)

    lr_actual = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>9.4f} | {val_acc:>7.1%} | {lr_actual:>8.2e}")

    # ── Early stopping ────────────────────────────────────────────────────
    if val_loss < mejor_val_loss:
        mejor_val_loss    = val_loss
        epocas_sin_mejora = 0
        torch.save(model.state_dict(), RUTA_CKPT)
        print(f"         Mejor modelo guardado (val_loss={val_loss:.4f})")
    else:
        epocas_sin_mejora += 1
        if epocas_sin_mejora >= PATIENCE:
            print(f"\nEarly stopping activado: sin mejora en {PATIENCE} épocas.")
            break

print(f"\nEntrenamiento finalizado. Mejor val_loss: {mejor_val_loss:.4f}")

---

## Bloque 7: Curvas de aprendizaje

Las curvas de aprendizaje son el **diagnóstico clínico** del entrenamiento. Permiten detectar tres patrones:

- **Buen entrenamiento:** Ambas curvas bajan juntas y convergen. La brecha entre train y val es pequeña.
- **Overfitting:** La pérdida de train sigue bajando pero la de val se estanca o sube. El modelo memorizó el dataset.
- **Underfitting:** Ambas pérdidas se quedan altas. El modelo no tiene capacidad suficiente o entrenó poco.

Con Transfer Learning y datasets pequeños, el **overfitting** es el riesgo más común.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, len(historial['train_loss']) + 1)

# ── Pérdida ───────────────────────────────────────────────────────────────────
ax1.plot(epocas, historial['train_loss'], 'b-o', markersize=4, label='Entrenamiento')
ax1.plot(epocas, historial['val_loss'],   'r-o', markersize=4, label='Validación')
ax1.set_title('Pérdida (CrossEntropyLoss) por época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2.plot(epocas, [a*100 for a in historial['train_acc']], 'b-o', markersize=4, label='Entrenamiento')
ax2.plot(epocas, [a*100 for a in historial['val_acc']],   'r-o', markersize=4, label='Validación')
ax2.set_title('Precisión (Accuracy) por época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 105)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — Clasificador de Billetes PYG', fontsize=13)
plt.tight_layout()
plt.show()

# ── Diagnóstico automático ────────────────────────────────────────────────────
brecha = historial['train_acc'][-1] - historial['val_acc'][-1]
print('Diagnóstico del entrenamiento:')
print(f'  Accuracy train final:  {historial["train_acc"][-1]:.1%}')
print(f'  Accuracy val final:    {historial["val_acc"][-1]:.1%}')
print(f'  Brecha train-val:      {brecha:.1%}')
if brecha > 0.20:
    print("  Diagnóstico: OVERFITTING — el modelo memorizó el dataset.")
    print("  Acción sugerida: más fotos diversas, aumentar Dropout, más augmentation.")
elif historial['val_acc'][-1] < 0.65:
    print("  Diagnóstico: UNDERFITTING — el modelo no aprendió suficiente.")
    print("  Acción sugerida: más épocas, verificar calidad de las fotos y etiquetas.")
else:
    print("  Diagnóstico: Entrenamiento razonable.")

---

## Bloque 8: Evaluación — Matriz de confusión y métricas

### ¿Por qué no alcanza con el accuracy?

Supongamos que el modelo tiene 90% de accuracy. Suena bien, pero:
- Si el 10% de errores es siempre confundir **50k con 100k**, en una terminal real eso es un problema financiero grave.
- Si el 10% de errores está uniformemente distribuido entre todas las clases, es mucho menos problemático.

La **matriz de confusión** muestra exactamente qué clases se confunden entre sí. En el contexto de billetes, queremos que la diagonal (predicciones correctas) tenga los valores más altos, y que los errores más graves (ej: 50k vs 100k) sean los más bajos.

### F1-Score para este problema

El F1-Score combina Precision y Recall. Para billetes, el **Recall** es especialmente importante: preferimos rechazar un billete válido (falso negativo) que aceptar un billete incorrecto (falso positivo). Sin embargo, rechazar demasiados válidos también genera fricciones con el usuario. El F1 balancea ambos.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# ── Cargar el mejor modelo ────────────────────────────────────────────────────
model.load_state_dict(torch.load(RUTA_CKPT, map_location=device))
print(f"Mejor modelo cargado desde: {RUTA_CKPT}")

# ── Recolectar todas las predicciones del set de validación ──────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs   = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# ── Matriz de confusión ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

# Normalizar para ver porcentajes (más informativo que valores absolutos)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Conteos absolutos
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax1, linewidths=0.5)
ax1.set_xlabel('Predicción del modelo')
ax1.set_ylabel('Denominación real')
ax1.set_title('Conteos absolutos')
ax1.tick_params(axis='x', rotation=45)

# Porcentajes por fila (recall por clase)
sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax2, linewidths=0.5, vmin=0, vmax=1)
ax2.set_xlabel('Predicción del modelo')
ax2.set_ylabel('Denominación real')
ax2.set_title('Porcentaje por fila (Recall por clase)')
ax2.tick_params(axis='x', rotation=45)

fig.suptitle('Matriz de Confusión — Terminal de Depósito PYG', fontsize=13)
plt.tight_layout()
plt.show()

# ── Reporte de métricas ───────────────────────────────────────────────────────
print('Reporte de clasificación:\n')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# ── Ranking de clases por F1-Score ────────────────────────────────────────────
f1s = f1_score(all_labels, all_preds, average=None)
print('Ranking de denominaciones por F1-Score (de más difícil a más fácil):')
for clase, f1 in sorted(zip(CLASS_NAMES, f1s), key=lambda x: x[1]):
    barra = '█' * int(f1 * 20)
    print(f'  {clase:<8} {barra:<20} {f1:.3f}')

---

## Bloque 9: Simulador de terminal de depósito

Este bloque simula exactamente lo que haría el software de una terminal de servicios real:

1. Recibe una imagen del billete (de la cámara de la ranura)
2. La preprocesa con las mismas transformaciones que usó en entrenamiento
3. Genera probabilidades para cada denominación
4. Aplica un **umbral de seguridad**: si la confianza máxima está por debajo del umbral, rechaza el billete y le pide al usuario que lo reintente

### ¿Por qué el umbral de seguridad es crucial?

Sin umbral, la terminal siempre aceptaría un billete, aunque el modelo esté muy inseguro (ej: 30% de confianza). En una terminal financiera real, una confusión de denominaciones es un error de auditoría. El umbral del 85% significa: "Solo acepto si estoy muy seguro, de lo contrario devuelvo el billete".

In [ ]:
UMBRAL_SEGURIDAD = 0.85  # 85% de confianza mínima para aceptar el depósito

def terminal_deposito_pyg(ruta_imagen: str, modelo, clases: list,
                           umbral: float = UMBRAL_SEGURIDAD):
    """
    Simula el software de la ranura de depósito de una terminal de servicios.
    
    Args:
        ruta_imagen: Ruta a la foto del billete
        modelo: Modelo PyTorch entrenado (en modo eval)
        clases: Lista de nombres de clase
        umbral: Confianza mínima para aceptar (0-1)
    
    Returns:
        dict con 'clase', 'confianza', 'aceptado'
    """
    if not os.path.exists(ruta_imagen):
        print(f"ERROR: No se encontró el archivo '{ruta_imagen}'")
        return None

    # ── Cargar y preprocesar imagen ───────────────────────────────────────
    img_pil = Image.open(ruta_imagen).convert('RGB')
    tensor  = transform_val(img_pil).unsqueeze(0).to(device)

    # ── Inferencia ────────────────────────────────────────────────────────
    modelo.eval()
    with torch.no_grad():
        logits = modelo(tensor)
        probs  = F.softmax(logits, dim=1).squeeze().cpu()

    confianza_max, idx_pred = torch.max(probs, 0)
    confianza_pct = confianza_max.item()
    clase_pred    = clases[idx_pred.item()]
    aceptado      = confianza_pct >= umbral

    # ── Visualización ─────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Panel izquierdo: imagen del billete + veredicto
    ax1.imshow(img_pil)
    ax1.axis('off')
    color_borde = '#2ecc71' if aceptado else '#e74c3c'
    estado_txt  = 'DEPÓSITO ACEPTADO ✅' if aceptado else 'DEPÓSITO RECHAZADO ❌'
    ax1.set_title(
        f"Billete detectado: {clase_pred} PYG\n"
        f"Confianza: {confianza_pct:.1%}\n"
        f"{estado_txt}",
        fontsize=11, color=color_borde, fontweight='bold'
    )
    for spine in ax1.spines.values():
        spine.set_edgecolor(color_borde)
        spine.set_linewidth(3)

    # Panel derecho: distribución de probabilidades
    probs_np = probs.numpy()
    colores  = ['#3498db'] * len(clases)
    colores[idx_pred.item()] = '#2ecc71' if aceptado else '#e74c3c'

    bars = ax2.barh(clases, probs_np * 100, color=colores, edgecolor='white')
    ax2.axvline(x=umbral * 100, color='orange', linestyle='--',
                linewidth=2, label=f'Umbral ({umbral:.0%})')
    ax2.set_xlabel('Confianza (%)')
    ax2.set_title('Distribución de probabilidades\npor denominación')
    ax2.set_xlim(0, 110)
    ax2.legend(fontsize=9)

    for bar, prob in zip(bars, probs_np):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{prob:.1%}', va='center', fontsize=9)

    plt.suptitle('SISTEMA DE DEPÓSITO AUTOMÁTICO — FCyT / UNCA', fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── Log de terminal ───────────────────────────────────────────────────
    print('\n' + '='*48)
    print('       SISTEMA DE DEPÓSITO AUTOMÁTICO — UNCA')
    print('='*48)
    if aceptado:
        print(f' RESULTADO: Billete de {clase_pred} PYG')
        print(f' ESTADO:    DEPÓSITO ACEPTADO ✅')
        print(f' CONFIANZA: {confianza_pct:.2%}')
    else:
        print(f' RESULTADO: No se puede validar con certeza')
        print(f' ESTADO:    DEPÓSITO RECHAZADO ❌')
        print(f' MOTIVO:    Confianza {confianza_pct:.2%} < umbral {umbral:.0%}')
        print(f' ACCIÓN:    Reintente con mejor iluminación o billete más liso.')
    print('='*48)

    return {'clase': clase_pred, 'confianza': confianza_pct, 'aceptado': aceptado}


# ── Ejemplo de uso ────────────────────────────────────────────────────────────
# Reemplazá esta ruta con una foto real de un billete para probar:
# resultado = terminal_deposito_pyg('mi_billete_prueba.jpg', model, CLASS_NAMES)
print("Función 'terminal_deposito_pyg' definida y lista.")
print("Ejemplo de uso:")
print("  resultado = terminal_deposito_pyg('foto_billete.jpg', model, CLASS_NAMES)")

---

## Bloque 10: Exportación del modelo

En producción, el modelo entrenado se guarda en un archivo `.pth` (los pesos) junto con los metadatos del experimento. Para usar el modelo en otro script o en la terminal real, solo necesitás:
1. Tener la arquitectura definida (`crear_modelo_pyg`)
2. Cargar los pesos con `load_state_dict`
3. Llamar a `model.eval()` antes de cualquier inferencia

Guardar los metadatos es una práctica profesional: si en 6 meses necesitás reproducir el experimento o saber con qué datos fue entrenado, esa información estará documentada.

In [ ]:
RUTA_MODELO_FINAL = 'modelo_terminal_guaranies.pth'

# ── Guardar el mejor modelo ───────────────────────────────────────────────────
# Cargamos el checkpoint guardado durante el entrenamiento (el de menor val_loss)
model.load_state_dict(torch.load(RUTA_CKPT, map_location=device))
torch.save(model.state_dict(), RUTA_MODELO_FINAL)

# ── Guardar metadatos del experimento ─────────────────────────────────────────
metadatos = {
    'fecha_entrenamiento': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'arquitectura':        'MobileNetV2 + Clasificador personalizado',
    'estrategia':          'Feature Extraction (capas congeladas)',
    'clases':              CLASS_NAMES,
    'num_clases':          NUM_CLASSES,
    'img_size':            IMG_SIZE,
    'batch_size':          BATCH_SIZE,
    'learning_rate':       LEARNING_RATE,
    'epocas_entrenadas':   len(historial['train_loss']),
    'mejor_val_loss':      round(mejor_val_loss, 4),
    'val_accuracy_final':  round(historial['val_acc'][-1], 4),
    'umbral_seguridad':    UMBRAL_SEGURIDAD,
    'seed':                SEED,
    'imagenet_mean':       IMAGENET_MEAN,
    'imagenet_std':        IMAGENET_STD
}

with open('metadatos_terminal_pyg.json', 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print(f'Modelo final guardado en:  {RUTA_MODELO_FINAL}')
print(f'Metadatos guardados en:    metadatos_terminal_pyg.json')
print('\nResumen del experimento:')
for clave, valor in metadatos.items():
    if clave not in ('imagenet_mean', 'imagenet_std'):
        print(f'  {clave}: {valor}')

print('\nPara cargar el modelo en otro script:')
print('  model = crear_modelo_pyg(num_clases=NUM_CLASSES)')
print(f'  model.load_state_dict(torch.load("{RUTA_MODELO_FINAL}"))')
print('  model.eval()')

---

# Tarea: El Desafío de la Terminal

## Parte A: Mejora iterativa del dataset

Después del primer entrenamiento, revisá la matriz de confusión y respondé:

1. ¿Qué par de denominaciones confunde más el modelo? ¿Cuál creés que es la razón visual?
2. Agregá 10 fotos más específicamente de esas denominaciones problemáticas, con mayor diversidad de fondos e iluminación.
3. Re-entrenó y compará las matrices de confusión antes y después. ¿Mejoró?

## Parte B: Análisis del umbral de seguridad

El umbral del 85% no es arbitrario. Usá la función `terminal_deposito_pyg()` con 10 fotos de validación y completá esta tabla:

| Imagen | Denominación real | Predicción | Confianza | ¿Aceptado al 85%? | ¿Correcto? |
|---|---|---|---|---|---|
| 1 | | | | | |
| ... | | | | | |

Luego respondé: ¿qué pasa con el negocio si bajás el umbral al 60%? ¿Y si lo subís al 95%?

## Parte C: Reflexión Data-Centric (para el informe)

1. **Calidad vs. cantidad:** Con los mismos datos, ¿vale más tener 50 fotos variadas o 100 fotos similares entre sí?
2. **Dominio del problema:** ¿Por qué el modelo podría fallar con billetes de otro país aunque sean del mismo valor (ej: billetes dolarizados)?
3. **Despliegue real:** Si esta terminal se instalara en Asunción, ¿qué condiciones del entorno habría que agregar al dataset que quizás no consideraron en el aula?

---

## Tabla resumen de conceptos

| Concepto | ¿Qué hace? | ¿Cuándo importa? |
|---|---|---|
| **Transfer Learning** | Reutiliza pesos de un modelo pre-entrenado | Cuando tenemos pocos datos |
| **Feature Extraction** | Congela el extractor, entrena solo el clasificador | Con datasets muy pequeños (<500 imágenes) |
| **Data Augmentation** | Genera variantes artificiales de cada imagen | Siempre, especialmente con pocos datos |
| **Early Stopping** | Detiene si val_loss no mejora en N épocas | Evita overfitting |
| **Umbral de confianza** | Rechaza predicciones inseguras | En aplicaciones críticas (financiero, médico) |
| **Matriz de confusión** | Muestra qué clases se confunden entre sí | Para diagnosticar errores sistemáticos |
| **Data-Centric AI** | Prioriza calidad de datos sobre arquitectura | En producción real |
| **Gradient Clipping** | Limita magnitud de gradientes | Estabiliza el entrenamiento |